# 01. 기초: 대화 단위 분할과 턴별 전환 확률 예측

## 학습 목표

- 영업 대화를 `conversation -> turn -> state` 구조로 표현한다.
- 같은 대화의 턴이 학습·평가 양쪽에 섞이지 않도록 **대화 ID 단위**로 분할한다.
- NumPy만으로 로지스틱 회귀를 학습하고 accuracy, log-loss, Brier score, ECE를 비교한다.
- 불균형 자료에서는 accuracy 하나만으로 좋은 확률 예측기라고 결론 내릴 수 없음을 확인한다.

> **Toy reproduction 주의:** 아래 데이터, 특징, 모델, 수치는 모두 교육용 합성 예제다. 논문 및 후속 공개 모델·데이터 아티팩트를 사용하지 않으며, 보고된 96.7% 정확도나 43.2% 전환 향상을 재현하지 않는다. 외부 API, 네트워크, GPU를 사용하지 않는다.

실행 환경: Python 3 + NumPy. 모든 난수는 고정 seed를 사용한다.

In [ ]:
import numpy as np

SEED = 250323303
rng = np.random.default_rng(SEED)

def sigmoid(z):
    z = np.clip(z, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-z))

def binary_metrics(y, p, threshold=0.5, bins=10):
    # 확률 자체의 품질을 보기 위해 log-loss와 Brier score를 함께 계산한다.
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    accuracy = np.mean((p >= threshold) == y)
    log_loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    brier = np.mean((p - y) ** 2)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p >= lo) & ((p < hi) if hi < 1.0 else (p <= hi))
        if np.any(mask):
            ece += np.mean(mask) * abs(np.mean(p[mask]) - np.mean(y[mask]))
    return {"accuracy": accuracy, "log_loss": log_loss, "brier": brier, "ece": ece}

def print_metrics(name, values):
    text = ", ".join(f"{key}={value:.4f}" for key, value in values.items())
    print(f"{name:24s} | {text}")

print("NumPy", np.__version__, "| seed", SEED)

## 1. 합성 대화 만들기

각 고객에게 관측되지 않는 잠재 구매 의도(`latent_intent`)가 있다고 가정한다. 상담 중에는 관여도, 이의 제기, 사용한 영업 기법과 지금까지의 누적 신호만 관측한다. 최종 전환 레이블은 대화가 끝난 뒤 한 번 정해진다.

중요한 점은 한 대화의 모든 턴이 같은 레이블을 공유한다는 사실이다. 턴을 무작위로 나누면 거의 동일한 대화가 학습과 평가에 동시에 들어가는 **그룹 누수(group leakage)**가 생긴다. 따라서 먼저 대화 ID를 나누고 그 뒤에 턴을 펼친다.

In [ ]:
def make_conversations(n_conversations=900):
    rows, labels, conversation_ids = [], [], []
    for cid in range(n_conversations):
        latent_intent = rng.normal()
        n_turns = int(rng.integers(4, 11))
        engagement = 0.25 * latent_intent + rng.normal(scale=0.7)
        engagements, objections, techniques = [], [], []
        conversation_rows = []
        for turn in range(n_turns):
            # 기법 선택은 현재까지 관측한 상태에만 의존한다. 정답 레이블은 보지 않는다.
            technique = float((engagement < 0.0) or (turn >= n_turns // 2))
            objection_p = sigmoid(-0.8 * latent_intent - 0.35 * engagement)
            objection = float(rng.random() < objection_p)
            engagement = (
                0.58 * engagement + 0.28 * latent_intent
                + 0.18 * technique - 0.32 * objection + rng.normal(scale=0.45)
            )
            engagements.append(engagement)
            objections.append(objection)
            techniques.append(technique)
            progress = (turn + 1) / n_turns
            # 현재 턴까지 이용 가능한 상태만 특징으로 사용한다.
            features = [
                progress, engagement, objection, technique,
                np.mean(engagements), np.mean(objections),
            ]
            conversation_rows.append((cid, turn + 1, n_turns, features))
        final_score = (
            -1.35 + 0.95 * latent_intent + 0.45 * engagements[-1]
            + 0.12 * np.mean(techniques) - 0.35 * np.mean(objections)
            + rng.normal(scale=0.65)
        )
        converted = float(rng.random() < sigmoid(final_score))
        for cid_, turn, total, features in conversation_rows:
            conversation_ids.append(cid_)
            rows.append([turn, total, *features])
            labels.append(converted)
    return np.asarray(rows, float), np.asarray(labels, float), np.asarray(conversation_ids, int)

rows, y, conversation_id = make_conversations()
all_ids = np.unique(conversation_id)
shuffled_ids = rng.permutation(all_ids)
cut = int(0.7 * len(shuffled_ids))
train_ids, test_ids = shuffled_ids[:cut], shuffled_ids[cut:]
train_mask = np.isin(conversation_id, train_ids)
test_mask = np.isin(conversation_id, test_ids)

assert set(train_ids).isdisjoint(set(test_ids)), "대화 ID 누수가 없어야 한다."
assert np.all(train_mask ^ test_mask), "모든 턴은 정확히 한 split에 속해야 한다."
print(f"대화 {len(all_ids):,}개, 턴 {len(y):,}개")
print(f"학습 대화 {len(train_ids):,}개 / 평가 대화 {len(test_ids):,}개")
print(f"평가 대화 전환율: {np.mean(y[test_mask]):.3f} (턴 가중 표시)")

## 2. 대화 상태로 확률 예측하기

첫 두 열은 `현재 턴`, `전체 턴 수`이며 모델 입력에서는 제외한다. 나머지 여섯 상태 특징을 학습 split의 평균·표준편차로만 표준화한다. 아래 경사하강은 교육용 최소 구현이다. 실제 시스템에서는 정규화, 하이퍼파라미터 탐색, 불확실성 추정과 시간 순서 검증이 추가로 필요하다.

In [ ]:
X = rows[:, 2:]
mean = X[train_mask].mean(axis=0)
std = X[train_mask].std(axis=0) + 1e-8
X_scaled = (X - mean) / std
X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled])

def fit_logistic(X_train, y_train, steps=1800, learning_rate=0.08, l2=2e-3):
    weights = np.zeros(X_train.shape[1])
    penalty_mask = np.r_[0.0, np.ones(X_train.shape[1] - 1)]
    for _ in range(steps):
        prediction = sigmoid(X_train @ weights)
        gradient = X_train.T @ (prediction - y_train) / len(y_train)
        gradient += l2 * penalty_mask * weights
        weights -= learning_rate * gradient
    return weights

weights = fit_logistic(X_design[train_mask], y[train_mask])
test_probability = sigmoid(X_design[test_mask] @ weights)
test_y = y[test_mask]
test_rows = rows[test_mask]

# 비교 1: 전부 비전환으로 분류. accuracy만 높을 수 있지만 유효한 확률을 내지 않는다.
all_negative_probability = np.full_like(test_y, 1e-7)
# 비교 2: 학습 대화의 전환율만 항상 출력하는 확률 기준선.
train_prevalence = np.mean(y[train_mask])
prevalence_probability = np.full_like(test_y, train_prevalence)

print_metrics("항상 비전환(accuracy 함정)", binary_metrics(test_y, all_negative_probability))
print_metrics("학습 전환율 기준선", binary_metrics(test_y, prevalence_probability))
print_metrics("턴별 상태 모델", binary_metrics(test_y, test_probability))

# 대화 초반과 마지막 턴을 따로 보면 정보가 누적되는 효과를 확인할 수 있다.
early = test_rows[:, 0] == 1
final = test_rows[:, 0] == test_rows[:, 1]
print_metrics("상태 모델 - 첫 턴", binary_metrics(test_y[early], test_probability[early]))
print_metrics("상태 모델 - 마지막 턴", binary_metrics(test_y[final], test_probability[final]))

assert np.all(np.isfinite(test_probability))
assert np.all((test_probability >= 0.0) & (test_probability <= 1.0))
assert 0.05 < np.mean(test_y) < 0.5, "accuracy 함정을 관찰할 수 있는 불균형이어야 한다."
assert binary_metrics(test_y, test_probability)["log_loss"] < binary_metrics(test_y, all_negative_probability)["log_loss"]
print("검증 통과: 누수 없는 분할, 유효한 확률, 기준선보다 낮은 log-loss")

## 3. 의도적으로 만든 누수 실패 사례

위 정상 모델의 입력은 **현재 턴까지의 prefix 신호만** 사용하고, split도 대화 ID 단위다. 이제 최종 결과인 `outcome`을 입력 열에 몰래 붙여 같은 평가를 반복한다. 이는 실시간 추론 시 아직 존재하지 않는 답을 보는 명백한 target leakage다.

> 이 점검은 논문 발표 뒤 공개된 Hugging Face 학습 스크립트를 감사하면서 발견할 수 있는 누수 위험에서 착안한 교육 예제다. 아래 합성 실험은 그 후속 스크립트나 arXiv v1이 실제로 동일한 누수를 사용했다는 증명이 아니다. 코드 버전·데이터 스키마·실행 경로를 별도로 재현 감사해야 한다.

In [ ]:
# 절대 실서비스 모델에 넣어서는 안 되는 열: 최종 정답 y 자체.
X_leaked = np.column_stack([X_design, y])
leaked_weights = fit_logistic(X_leaked[train_mask], y[train_mask], steps=1800)
leaked_probability = sigmoid(X_leaked[test_mask] @ leaked_weights)
clean_metrics = binary_metrics(test_y, test_probability)
leaked_metrics = binary_metrics(test_y, leaked_probability)
print_metrics("정상 prefix-only 모델", clean_metrics)
print_metrics("정답 누수 모델(무효)", leaked_metrics)
print(f"겉보기 accuracy 상승: {leaked_metrics['accuracy'] - clean_metrics['accuracy']:+.3f}")

assert leaked_metrics["accuracy"] > 0.98
assert leaked_metrics["accuracy"] > clean_metrics["accuracy"] + 0.20
print("경고: 이 상승은 알고리즘 개선이 아니라 추론 시 알 수 없는 정답을 입력한 결과다.")

## 결과 읽기

- `항상 비전환`은 비전환이 많은 데이터에서 accuracy가 그럴듯해 보일 수 있다. 그러나 실제 전환에 거의 0의 확률을 주므로 log-loss가 매우 커진다.
- Brier score는 확률과 실제 레이블의 제곱 오차, ECE(Expected Calibration Error)는 구간별 평균 확률과 실제 빈도의 차이를 요약한다. 낮을수록 좋다.
- 첫 턴보다 마지막 턴에 더 많은 정보가 있지만, 이것만으로 실시간 의사결정의 사업 효과가 입증되지는 않는다. 확률 예측과 행동 최적화는 다른 문제다.
- 누수 모델의 거의 완벽한 정확도는 성공이 아니라 평가 파이프라인 실패다. 각 특징에 대해 ‘그 턴 시점에 실제로 알 수 있었는가?’를 검사해야 한다.
- 실무에서는 고객·계정·영업 담당자처럼 공유 원인이 있는 단위까지 묶어 분할하고, 시간 순서가 있는 경우 과거로 학습해 미래를 평가해야 한다.

다음 노트북에서는 ‘확률을 출력하는 예측기’와 ‘환경을 바꾸는 행동 정책’을 명시적으로 구분한다.